# Random State Handling in Obsidian

This notebook demonstrates the new and old approaches to random number generation (RNG) control in Obsidian. Proper RNG control is essential for:

- **Reproducibility**: Ensure experiments can be replicated exactly
- **Debugging**: Track down issues with deterministic behavior
- **Statistical Integrity**: Control variance sources in stochastic processes

## Table of Contents
1. [New RNG Control (Recommended)](#new-rng)
2. [Old RNG Control (Legacy)](#old-rng)
3. [Comparison & Best Practices](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import torch
import obsidian
from obsidian import Campaign, create_rng_manager
from obsidian.parameters import ParamSpace, Param_Continuous, Param_Categorical, Target
from obsidian.experiment import Simulator
from obsidian.experiment.benchmark import shifted_parab

print(f"Obsidian version: {obsidian.__version__}")

In [ ]:
# Setup parameter space and target for examples
X_space = ParamSpace([
    Param_Continuous('Temperature', 0, 100),
    Param_Continuous('Pressure', -20, 20),
    Param_Continuous('Time', 1, 10),
    Param_Categorical('Catalyst', ['A', 'B', 'C'])
])

target = Target(name='Yield', f_transform='Standard', aim='max')
simulator = Simulator(X_space, shifted_parab, name='Yield', eps=0.05)

<a id="new-rng"></a>
## 1. New RNG Control (Recommended)

The new RNG control system uses `RNGManager` to coordinate random number generation across NumPy, PyTorch, and Python's random module. This provides:

- **Unified control** across all RNG sources
- **State persistence** for saving/loading experiments
- **Flexible seeding** with independent RNG instances
- **Temporary seed override** for specific operations

### 1.1 Creating Independent RNG Managers

Use `create_rng_manager()` to create independent RNG instances. This is the **recommended approach** for running multiple experiments in parallel.

In [ ]:
# Create an RNG manager with explicit seed
rng1 = create_rng_manager(seed=12345)

# Generate random numbers using NumPy
random_nums_np = rng1.np_rng.random(5)
print(f"NumPy random numbers: {random_nums_np}")

# Generate random numbers using PyTorch
random_nums_torch = torch.rand(5, generator=rng1.torch_rng)
print(f"PyTorch random numbers: {random_nums_torch}")

# Create another RNG with the same seed - should produce identical results
rng2 = create_rng_manager(seed=12345)
random_nums_np2 = rng2.np_rng.random(5)
print(f"\nWith same seed (12345): {np.allclose(random_nums_np, random_nums_np2)}")

# Create RNG with different seed - produces different results
rng3 = create_rng_manager(seed=67890)
random_nums_np3 = rng3.np_rng.random(5)
print(f"With different seed (67890): {np.allclose(random_nums_np, random_nums_np3)}")

### 1.2 Using RNG with Campaign (Automatic Management)

When creating a Campaign with a seed, it automatically creates and manages an `RNGManager` internally.

In [ ]:
# Create campaign with seed - automatically creates RNGManager
campaign1 = Campaign(X_space, target, seed=999)

# Initialize design of experiments
X_init = campaign1.initialize(m_initial=5, method='LHS')
print(f"Initial design shape: {X_init.shape}")
print(f"\nFirst 3 rows:\n{X_init.head(3)}")

# Campaign has an RNG manager
print(f"\nCampaign has RNG manager: {hasattr(campaign1, 'rng')}")
print(f"RNG seed: {campaign1.rng.seed}")

### 1.3 Sharing RNG Between Components (Advanced)

You can create an RNG and share it across multiple components for coordinated randomness.

**Important for Simulator Users:** The `Simulator` class does **not** support state save/load at this time. Therefore, when using a simulator, it's **highly recommended** to share an RNG between the simulator and campaign. This ensures both can be restored together from the campaign's saved state.

### 1.2.5 How Components Create Their Own RNG

**Important:** Many Obsidian components can create their own `RNGManager` if you pass an integer seed. Understanding this is crucial for proper RNG control.

In [ ]:
# Components that accept RNG parameters:
# 1. Campaign(seed=...) or Campaign(rng=...)
# 2. Simulator(rng=...) - accepts int, Generator, or RNGManager
# 3. BayesianOptimizer(seed=..., rng=..., fix_random_state=...)
# 4. ExpDesigner(seed=..., rng=...)

# Example: Simulator creates its own RNG from an integer seed
sim_from_int = Simulator(X_space, shifted_parab, eps=0.05, rng=42)
print("Simulator created internal RNG with numpy Generator")
print(f"Type: {type(sim_from_int.rng)}")

# Example: Campaign creates its own RNG from an integer seed  
campaign_from_int = Campaign(X_space, target, seed=123)
print("\nCampaign created internal RNGManager")
print(f"Has RNG: {hasattr(campaign_from_int, 'rng')}")
print(f"RNG seed: {campaign_from_int.rng.seed}")

print("\n✓ Each component manages its own random state independently")

### 1.2.6 Understanding the Random State in Optimizers

The `fix_random_state` parameter in `BayesianOptimizer` controls whether **`suggest()` operations** produce deterministic results.

**Default: `fix_random_state=True` (Deterministic)**
- Uses the **same seed** for all `suggest()` operations
- Multiple calls to `suggest()` with the same fitted model produce **identical results**
- Recommended for most cases
- **Note:** `fit()` results are the same regardless of this setting

**Alternative: `fix_random_state=False` (Stochastic)**
- Creates an internal generator that produces **new seeds** for each `suggest()` call
- Multiple `suggest()` calls produce **different results** even for the same
  model (explores different optimization restarts)
- Useful for exploration or generating diverse solutions

**Important:** When creating a `Campaign` directly (without a custom optimizer), the internally created `BayesianOptimizer` uses the default `fix_random_state=True`. You can bypass this with `campaign.suggest(manual_seed=...)`.

In [ ]:
from obsidian.optimizer import BayesianOptimizer

# Create two optimizers with fix_random_state=True (default)
opt1_fixed = BayesianOptimizer(X_space, seed=777, fix_random_state=True)

# Generate test data
X_init_opt = campaign1.initialize(m_initial=5, method='LHS')
y_init_opt = simulator.simulate(X_init_opt)
Z_opt = pd.concat([X_init_opt, y_init_opt], axis=1)

# Fit the optimizer
opt1_fixed.fit(Z_opt, target)

# Make multiple suggestions - should be IDENTICAL with fix_random_state=True
X_suggest1, _ = opt1_fixed.suggest(m_batch=2, acquisition=['EI'])
X_suggest2, _ = opt1_fixed.suggest(m_batch=2, acquisition=['EI'])

print("fix_random_state=True (Default):")
print(f"Multiple suggest() calls are identical: {X_suggest1.equals(X_suggest2)}")

# Now try with fix_random_state=False
opt2_stoch = BayesianOptimizer(X_space, seed=888, fix_random_state=False)
opt2_stoch.fit(Z_opt, target)

# Make multiple suggestions - should be DIFFERENT with fix_random_state=False
X_suggest3, _ = opt2_stoch.suggest(m_batch=2, acquisition=['EI'])
X_suggest4, _ = opt2_stoch.suggest(m_batch=2, acquisition=['EI'])

print("\nfix_random_state=False:")
print(f"Multiple suggest() calls are identical: {X_suggest3.equals(X_suggest4)}")

### 1.2.7 Bypassing Deterministic Behavior

There are **two ways** to introduce variation when using optimizers:

1. **Create custom optimizer with `fix_random_state=False` and pass it to
   campaign at instantiation** (affects all operations)
2. **Use `manual_seed` parameter** in `suggest()` (temporary, one-time override)

**Note:** `Campaign` does **not** accept a `fix_random_state` parameter.

```python
campaign = Campaign(X_space, target, seed=123)
```
When you create a Campaign with just a seed, it internally creates a `BayesianOptimizer` with the default `fix_random_state=True`. 

**To use stochastic behavior with Campaign:**
```python
# Create optimizer with fix_random_state=False
custom_opt = BayesianOptimizer(X_space, seed=123, fix_random_state=False)

# Pass it to Campaign
campaign = Campaign(X_space, target, optimizer=custom_opt)
```

**Or for temporary variation, use `manual_seed`:**
```python
campaign = Campaign(X_space, target, seed=123)  # Still deterministic by default
X_suggest, _ = campaign.suggest(m_batch=3, manual_seed=999)  # One-time override
```

In [ ]:
# Method 1: Create custom optimizer with fix_random_state=False
custom_opt = BayesianOptimizer(X_space, seed=555, fix_random_state=False)
campaign_stochastic = Campaign(X_space, target, optimizer=custom_opt)

X_init_stoch = campaign_stochastic.initialize(m_initial=5, method='LHS')
y_init_stoch = simulator.simulate(X_init_stoch)
campaign_stochastic.add_data(pd.concat([X_init_stoch, y_init_stoch], axis=1))
campaign_stochastic.fit()

print("Method 1: fix_random_state=False")
print("Multiple suggest() calls will use different seeds internally")
X_suggest_stoch1, _ = campaign_stochastic.suggest(m_batch=2, acquisition=['EI'])
X_suggest_stoch2, _ = campaign_stochastic.suggest(m_batch=2, acquisition=['EI'])
print(f"Suggestions are different: {not X_suggest_stoch1.equals(X_suggest_stoch2)}")

# Method 2: Use manual_seed with regular Campaign
print("\n" + "="*60)
campaign_regular = Campaign(X_space, target, seed=666)  # Uses fix_random_state=True by default
X_init_reg = campaign_regular.initialize(m_initial=5, method='LHS')
y_init_reg = simulator.simulate(X_init_reg)
campaign_regular.add_data(pd.concat([X_init_reg, y_init_reg], axis=1))
campaign_regular.fit()

print("Method 2: Use manual_seed parameter (temporary override)")
X_suggest_normal1, _ = campaign_regular.suggest(m_batch=2, acquisition=['EI'])
X_suggest_normal2, _ = campaign_regular.suggest(m_batch=2, acquisition=['EI'])
print(f"Without manual_seed, multiple suggest() are identical: {X_suggest_normal1.equals(X_suggest_normal2)}")

X_suggest_override, _ = campaign_regular.suggest(m_batch=2, acquisition=['EI'], manual_seed=9999)
print(f"With manual_seed, result can differ from default: {not X_suggest_normal1.equals(X_suggest_override)}")

print("\n" + "="*60)
print("Recommendation:")
print("  - Campaign uses fix_random_state=True by default (deterministic)")
print("  - Create custom optimizer with fix_random_state=False for persistent stochastic behavior")
print("  - Use campaign.suggest(manual_seed=...) for one-off variations (for example, if you are not happy with the suggested parameters)")

### 1.3 Create shared RNG

You can instantiate an `RNGManager` and share it across multiple components to ensure they use the same random state.

In [ ]:
# Create a shared RNG - RECOMMENDED PATTERN for simulators
shared_rng = create_rng_manager(seed=777)

# Use it with simulator (since simulator doesn't support save/load)
simulator_shared = Simulator(X_space, shifted_parab, eps=0.05, rng=shared_rng)

# Use it with campaign - both share the same RNG
campaign_shared = Campaign(X_space, target, rng=shared_rng)

print(f"Simulator NumPy RNG seed: {simulator_shared.rng.bit_generator.state['state']['state']}")
print(f"Campaign NumPy RNG seed: {campaign_shared.rng.np_rng.bit_generator.state['state']['state']}")
print(f"Campaign RNG seed: {campaign_shared.rng.seed}")
print(f"Same RNG object: {campaign_shared.rng is shared_rng}")

print("\n✓ When campaign state is saved/loaded, the shared RNG is restored")
print("✓ Simulator can continue using the restored RNG consistently")

### 1.4 Saving and Loading RNG State

RNG state can be saved and restored to continue experiments from exactly where you left off.

In [ ]:
# Create RNG and generate some random numbers
rng_original = create_rng_manager(seed=555)
before_save = rng_original.np_rng.random(3)
print(f"Before save: {before_save}")

# Save the RNG state
rng_state = rng_original.save_state()
print(f"\nState saved with seed: {rng_state['seed']}")

# Generate more numbers (advances the state)
after_save = rng_original.np_rng.random(3)
print(f"After save (original): {after_save}")

# Load state into a new RNG - it should continue from where we saved
from obsidian.rng import RNGManager
rng_restored = RNGManager.load_state(rng_state)

# Should produce the same numbers as 'after_save'
after_restore = rng_restored.np_rng.random(3)
print(f"After restore: {after_restore}")
print(f"\nState correctly restored: {np.allclose(after_save, after_restore)}")

<a id="old-rng"></a>
## 2. Old RNG Control (Legacy)

The old approach uses simple integer seeds passed to components. This is **deprecated** but still supported for backward compatibility.

### Limitations:
- No unified state management
- Less control over reproducibility
- Easy random state contamination due to accidental drawing random numbers outside of the intended flow

In [ ]:
# Enable old RNG control mode
obsidian.USE_OLD_RNG_CONTROL = True

# Create campaign with seed (old style)
campaign_old = Campaign(X_space, target, seed=999)

# Initialize design
X_init_old = campaign_old.initialize(m_initial=5, method='LHS')
print(f"Old RNG - Initial design shape: {X_init_old.shape}")

# Campaign does NOT have RNG manager in old mode
print(f"\nCampaign has RNG manager: {hasattr(campaign_old, 'rng') and campaign_old.rng is not None}")

# Re-enable new RNG control for rest of notebook
obsidian.USE_OLD_RNG_CONTROL = False
print("\n✓ Old RNG mode demonstrated. Switching back to new RNG control.")

<a id="comparison"></a>
## 3. Comparison & Best Practices

### Feature Comparison

| Feature | New RNG Control | Old RNG Control |
|---------|----------------|-----------------|
| **Unified RNG management** | ✅ Yes | ⚠️ Yes, but through global manual seed overriding |
| **State save/load** | ✅ Yes | ⚠️ No state, just overriding |
| **Independent RNG instances** | ✅ Yes | ❌ No |
| **Manual seed override** | ✅ Yes | ❌ No, just a global override |
| **Recommended for new code** | ✅ Yes | ❌ No |

### Best Practices

1. **Let Campaign handle everything automatically**
   ```python
   campaign = Campaign(X_space, target, seed=123)

2. **Share RNG between Simulator and Campaign if Simulator is needed**
   ```python
   # Simulator doesn't support save/load, so share RNG with campaign
   shared_rng = create_rng_manager(seed=777)
   sim = Simulator(X_space, func, rng=shared_rng)
   campaign = Campaign(X_space, target, rng=shared_rng)
   # When campaign is saved/loaded, RNG state is preserved for both
   ```

3. **Use `create_rng_manager()` for independent experiments**
   ```python
   rng1 = create_rng_manager(seed=42)
   rng2 = create_rng_manager(seed=43)
   # Each has independent random state
   ```

4. **Understand that components can create their own RNG**
   ```python
   # Simulator accepts int, Generator, or RNGManager
   sim = Simulator(X_space, func, rng=42)  # Creates internal RNG
   
   # Campaign accepts seed or RNGManager
   campaign = Campaign(X_space, target, seed=123)  # Creates internal RNG
   ```

5. **Use fix_random_state=True for reproducibility (default)**
   ```python
   # Deterministic suggest() - recommended for most use cases
   opt = BayesianOptimizer(X_space, seed=123, fix_random_state=True)
   opt.fit(data, target)
   X1, _ = opt.suggest(m_batch=3)
   X2, _ = opt.suggest(m_batch=3)
   # X1 and X2 are identical
   
   # Stochastic suggest() - for sensitivity analysis
   opt = BayesianOptimizer(X_space, seed=123, fix_random_state=False)
   opt.fit(data, target)
   X3, _ = opt.suggest(m_batch=3)
   X4, _ = opt.suggest(m_batch=3)
   # X3 and X4 are different
   ```

6. **Campaign doesn't accept fix_random_state directly**
   ```python
   # For stochastic behavior, create custom optimizer
   custom_opt = BayesianOptimizer(X_space, seed=123, fix_random_state=False)
   campaign = Campaign(X_space, target, optimizer=custom_opt)
   
   # Or use manual_seed for one-time override
   campaign = Campaign(X_space, target, seed=123)
   X, _ = campaign.suggest(m_batch=3, manual_seed=999)
   ```

All existing campaign saves should still work with the new RNG system and be
converted to use the new RNGManager internally. If you want to use the old
system, set `obsidian.USE_OLD_RNG_CONTROL = True` in your configuration and then
load your campaigns as usual.